# Site-specific weekly full-horizon oracle

Controlled benchmark for NTPLL, 1–7 March 2025. The common EV baseline is calculated only from pre-period Perfect history and is held fixed across Oracle, 24-h Perfect, and Persistence. This differs deliberately from the endogenous-baseline sensitivity runs. Passive building/PV affect retail only. Prices and attenuation are obtained from the initialized source MPC namespace. No automatic execution or executed-notebook copies.

Run `fixed_preperiod_baseline(...)` for the full MPC look-ahead index, then call `run_weekly_oracle(runtime, baseline, output_dir)`. A finite time-limit result is saved with its certified bound but is not marked accepted unless the requested 1% gap and optimal status are reached. Paper text is not changed.


In [ ]:
# Reusable functions: executing this cell does not launch an optimization.
from pathlib import Path
import importlib.util
import json
import hashlib
import time
import numpy as np
import pandas as pd
import holidays
import gurobipy as gp
from gurobipy import GRB
from scipy import sparse

REPO = Path('/Users/admin/Desktop/EV_program/Total Transfer/PowerFlex_Code/UPSCALeDEV_2024')
DT_H = 0.25

def fixed_preperiod_baseline(history_file, index, cutoff='2025-03-01'):
    """Historical 10-weekday/4-weekend-holiday hourly EV reference, kW.
    Only observations strictly before cutoff and strictly before target day
    are eligible. Event-hour exclusions and 45-day search mirror the MPC.
    No executed study-period dispatch is fed back into this controlled benchmark.
    """
    h = pd.read_csv(history_file, low_memory=False)
    h['Interval start'] = pd.to_datetime(h['Interval start'], errors='raise')
    h = h[h['Interval start'] < pd.Timestamp(cutoff)].drop_duplicates('Interval start', keep='last')
    if 'event hour_Base' not in h:
        raise ValueError('Common history must contain event hour_Base')
    h['Base [kWh]'] = pd.to_numeric(h['Base [kWh]'], errors='raise')
    idx = pd.DatetimeIndex(index)
    hol = holidays.US(years=sorted(set(idx.year) | set(h['Interval start'].dt.year)))
    def weekend(d):
        return d.weekday() >= 5 or d.date() in hol
    hourly = {}
    for day in idx.normalize().unique():
        count = 4 if weekend(day) else 10
        for hour in range(24):
            values = []
            for offset in range(1, 46):
                old = day - pd.Timedelta(days=offset)
                if old >= pd.Timestamp(cutoff) or weekend(old) != weekend(day):
                    continue
                g = h[(h['Interval start'].dt.normalize() == old) & (h['Interval start'].dt.hour == hour)]
                if g.empty or float(g['event hour_Base'].iloc[0]) != 0:
                    continue
                if len(g) != 4:
                    raise ValueError(f'Incomplete historical hour: {old} {hour}')
                values.append(float(g['Base [kWh]'].sum()))  # kWh / one hour = kW
                if len(values) == count:
                    break
            if not values:
                raise ValueError(f'No eligible baseline history for {day} hour {hour}')
            hourly[(day, hour)] = float(np.mean(values))
    return pd.Series([hourly[(t.normalize(), t.hour)] for t in idx], index=idx, name='baseline_kW')

def _weekly_ev_inputs(ev_file, index):
    """Same per-session interval filtering and charger classification as source MPC."""
    d = pd.read_csv(ev_file, low_memory=False)
    for col in ('Interval start', 'Interval end', 'Session start', 'Session end'):
        d[col] = pd.to_datetime(d[col], errors='raise')
    d = d[(d['Interval start'] >= index[0]) & (d['Interval start'] < index[-1] + pd.Timedelta(minutes=15))]
    d = d[(d['Interval start'] >= d['Session start']) & (d['Interval end'] <= d['Session end'])]
    d = d.sort_values(['Interval start', '10-digit UID']).reset_index(drop=True)
    if d.duplicated(['10-digit UID','Interval start']).any():
        raise ValueError('Duplicate session intervals must be resolved in source QC')
    upper = np.zeros(len(d))
    groups, records = [], []
    for uid, g in d.groupby('10-digit UID', sort=False):
        cap = 4.16 if ((g['Interval kWh'] > 1.664) | (g['Interval max demand kW'] > 6.656)).any() else 1.664
        inds = g.index.to_numpy()
        upper[inds] = cap
        target = min(float(g['Interval kWh'].sum()), cap * len(g))
        groups.append(inds)
        records.append({'session': str(uid), 'target_kwh': target, 'intervals': len(g), 'cap_kwh': cap})
    positions = pd.Series(np.arange(len(index)), index=index).reindex(d['Interval start']).to_numpy()
    if not np.isfinite(positions).all():
        raise ValueError('EV intervals are not on the model grid')
    A = sparse.csr_matrix((np.ones(len(d)), (positions.astype(int), np.arange(len(d)))), shape=(len(index),len(d)))
    S = sparse.lil_matrix((len(groups),len(d)))
    for i, ids in enumerate(groups):
        S[i, ids] = 1
    return A, S.tocsr(), upper, pd.DataFrame(records)


def _weekly_prices_exact_main(index):
    """Source MPC raw-day LMP and AS clock. Naive AS strings are California local."""
    result={k:[] for k in ('lmp_da','lmp_rt','as_ru_da','as_rd_da','as_sp_da','as_nsp_da','as_ru_rt','as_rd_rt','as_sp_rt','as_nsp_rt')}
    as_data={}
    for market,folder in [('da','AS_DAM'),('rt','AS_RTM')]:
        d=pd.read_csv(REPO/'2025Data'/folder/'AS_price_2025_clear.csv')
        text=d['datetime'].astype(str).str.strip()
        aware=text.str.contains(r'(?:Z|[+-]\d{2}:\d{2})$',regex=True)
        if aware.all():
            d['datetime']=pd.to_datetime(text,utc=True,errors='raise').dt.tz_convert('America/Los_Angeles').dt.tz_localize(None)
        elif (~aware).all():
            d['datetime']=pd.to_datetime(text,errors='raise')
        else:raise ValueError('Mixed timezone-aware/naive AS datetime')
        if d['datetime'].duplicated().any():raise ValueError('Duplicate AS clock')
        as_data[market]=d
    for day in index.normalize().unique():
        for market,folder,col in [('da','DA','MW'),('rt','FM','VALUE')]:
            paths=list((REPO/'2025Data/LMP/2025'/folder).glob(f'{day:%Y%m%d}*.csv'))
            if len(paths)!=1:raise ValueError(f'Nonunique price file: {market} {day}')
            d=pd.read_csv(paths[0]);d=d[d['LMP_TYPE']=='LMP'].sort_values('INTERVALSTARTTIME_GMT')
            a=d[col].to_numpy(float)
            if len(a)!=(24 if market=='da' else 288):raise ValueError('Incomplete LMP day')
            a=np.repeat(a,4)*.001 if market=='da' else a.reshape(-1,3).mean(axis=1)*.001
            result['lmp_'+market].extend(a)
        for market in ('da','rt'):
            d=as_data[market];d=d[d['datetime'].dt.normalize()==day]
            expected=pd.date_range(day,periods=24 if market=='da' else 96,freq='1h' if market=='da' else '15min')
            if not np.array_equal(d['datetime'].to_numpy(),expected.to_numpy()):
                raise ValueError(f'Nonchronological AS model clock: {market} {day}')
            scale=.001 if market=='da' else .001/DT_H
            for p,col in [('ru','RegUp'),('rd','RegDown'),('sp','Spin'),('nsp','NonSpin')]:
                a=d[col].to_numpy(float)*scale
                result['as_'+p+'_'+market].extend(np.repeat(a,4) if market=='da' else a)
    return {k:np.asarray(v,float) for k,v in result.items()}


def run_weekly_oracle(runtime, baseline, output_dir, *, site='NTPLL',
                      start='2025-03-01', end_exclusive='2025-03-08',
                      ev_file=None, btm_file=None, mip_gap=0.01,
                      time_limit_s=1800, threads=2, overwrite=False):
    """Solve a controlled full-horizon benchmark with fixed common EV baseline.
    runtime is the initialized source MPC namespace containing tariff/factor
    functions and current BESS parameters. No source notebook is mutated.
    DA is chosen with clairvoyance: the benchmark begins before DA submission.
    A finite incumbent is NOT a certified upper bound; ObjBound is reported.
    """
    index = pd.date_range(start, end_exclusive, freq='15min', inclusive='left')
    n = len(index)
    if not n or n % 96 or len(set(index.to_period('M'))) != 1 or index[0].day != 1:
        raise ValueError('Use complete days starting at billing-month start and contained in one month')
    output = Path(output_dir)
    output.mkdir(parents=True, exist_ok=True)
    if (output/'oracle_summary.csv').exists() and not overwrite:
        raise FileExistsError(f'Existing oracle summary is protected: {output}')
    baseline = pd.Series(baseline).reindex(index).to_numpy(float)
    if not np.isfinite(baseline).all() or np.min(baseline) < -1e-8:
        raise ValueError('Finite nonnegative common baseline required')
    ev_file = Path(ev_file or REPO/'2025Data/Site_Data_2025'/f'{site}_EV_2025_QC.csv')
    btm_file = Path(btm_file or REPO/'2025Data/Site_Data_2025'/f'{site}_BTM_2025_15min_QC.csv')
    A, S, ev_ub, ev_meta = _weekly_ev_inputs(ev_file, index)
    targets = ev_meta['target_kwh'].to_numpy(float)
    btm = pd.read_csv(btm_file)
    btm.index = pd.to_datetime(btm['Interval start'], errors='raise')
    if btm.index.duplicated().any():
        raise ValueError('Duplicate BTM timestamps')
    btm = btm.reindex(index)
    load = btm['p_load_kW'].to_numpy(float)
    pv = btm['p_PV_kW'].to_numpy(float)
    if not np.isfinite(load-pv).all():
        raise ValueError('Missing passive data')
    prices = _weekly_prices_exact_main(index)
    tou = np.asarray(runtime['get_sdge_dgr_tariff_profile'](index)[0], float)
    pd_mask = np.asarray(runtime['get_sdge_on_peak_mask'](index), bool)
    pd_rate = float(runtime['get_sdge_pd_rate'](index[0]))
    ncd_rate = float(runtime['c_NCD'])
    products = ('ru','rd','sp','nsp')
    alpha = dict(zip(products, [np.asarray(a,float) for a in runtime['_get_as_activation_profile'](index)]))
    P, E = float(runtime['P_BESS_max']), float(runtime['C_BESS'])
    lo, hi = float(runtime['SOC_BESS_min']), float(runtime['SOC_BESS_max'])
    eff = np.sqrt(float(runtime['gamma']))
    max_ev = np.asarray(A@ev_ub).ravel()/DT_H
    up = P+baseline
    down = np.maximum(P-baseline+max_ev,0)
    m_ev = 1.05*np.maximum.reduce([max_ev,np.abs(baseline),np.full(n,1e-6)])
    da_up = up.copy()
    da_down = down.copy()
    m=gp.Model('site_week_full_horizon_oracle')
    m.Params.MIPGap=float(mip_gap)
    m.Params.Threads=int(threads)
    m.Params.TimeLimit=float(time_limit_s)
    m.Params.LogFile=str(output/'oracle_gurobi.log')
    m.Params.MIPFocus=1
    m.Params.NumericFocus=1
    def v(name,lb=0,ub=GRB.INFINITY,size=n,binary=False):
        return m.addMVar(size,lb=lb,ub=ub,vtype=GRB.BINARY if binary else GRB.CONTINUOUS,name=name)
    x=v('ev_interval_kwh',ub=ev_ub,size=len(ev_ub))
    ev=v('ev_kW')
    m.addConstr(S@x==targets)
    m.addConstr(ev==(A@x)/DT_H)
    ch=v('bess_charge');dc=v('bess_discharge')
    chw=v('bess_wm_charge');dcw=v('bess_wm_discharge')
    chn=v('bess_nwm_charge');dcn=v('bess_nwm_discharge')
    b=v('bess_charge_direction',binary=True)
    soc=v('soc',lb=lo,ub=hi,size=n+1)
    m.addConstr(ch==chw+chn);m.addConstr(dc==dcw+dcn)
    m.addConstr(ch<=P*b);m.addConstr(dc<=P*(1-b))
    m.addConstr(soc[0]==.5);m.addConstr(soc[-1]==.5)
    m.addConstr(soc[1:]==soc[:-1]+DT_H/E*(eff*ch-dc/eff))
    for day in range(n//96):
        sl=slice(day*96,(day+1)*96)
        m.addConstr(DT_H*(ch[sl].sum()+dc[sl].sum())<=2*E*(hi-lo))
    gi=v('grid_interchange',lb=-GRB.INFINITY)
    m.addConstr(gi==load-pv+ev+ch-dc)
    da=v('energy_DA',lb=-da_down,ub=da_up)
    actual=v('energy_actual',lb=-down,ub=up)
    positive=v('energy_positive');negative=v('energy_negative')
    m.addConstr(positive>=actual);m.addConstr(negative>=-actual)
    cda={p:v('capacity_DA_'+p) for p in products}
    ca={p:v('capacity_actual_'+p) for p in products}
    dau=cda['ru']+cda['sp']+cda['nsp']
    au=ca['ru']+ca['sp']+ca['nsp']
    m.addConstr(dau<=da_up);m.addConstr(cda['rd']<=da_down)
    m.addConstr(positive+au<=up);m.addConstr(negative+ca['rd']<=down)
    for h in range(0,n,4):
        for off in (1,2,3):
            m.addConstr(da[h+off]==da[h])
            for p in products:m.addConstr(cda[p][h+off]==cda[p][h])
    ecw=v('ev_wm_charge');edw=v('ev_wm_discharge')
    ecn=v('ev_nwm_charge');edn=v('ev_nwm_discharge')
    be=v('ev_direction',binary=True)
    m.addConstr(ev-baseline==ecw-edw+ecn-edn)
    m.addConstr(ecw+ecn<=m_ev*be);m.addConstr(edw+edn<=m_ev*(1-be))
    deployment=sum(((-1 if p=='rd' else 1)*alpha[p]*ca[p] for p in products))
    obligation=actual+deployment
    nc=v('net_charge');nd=v('net_discharge');bn=v('net_charge_direction',binary=True)
    m.addConstr(nd>=obligation);m.addConstr(nd<=obligation+up*bn)
    m.addConstr(nc>=-obligation);m.addConstr(nc<=-obligation+down*(1-bn))
    m.addConstr(nc<=down*bn);m.addConstr(nd<=up*(1-bn))
    m.addConstr(nc-nd==chw-dcw+ecw-edw)
    if runtime.get('Direction_Aligned_AddOn',True):
        m.addConstr(au<=up*(1-bn));m.addConstr(ca['rd']<=down*bn)
    m.addConstr(soc[1:]>=lo+2*DT_H/(E*eff)*au)
    m.addConstr(soc[1:]<=hi-2*DT_H*eff/E*ca['rd'])
    ncd=m.addVar(lb=0,name='ncd_peak');pdpeak=m.addVar(lb=0,name='pd_peak')
    m.addConstr(gi<=ncd)
    if pd_mask.any():m.addConstr(gi[pd_mask]<=pdpeak)
    rev_ev=.4*DT_H*ev.sum()
    rev_e=DT_H*(prices['lmp_da']@da+prices['lmp_rt']@(actual-da+deployment))
    rev_c=sum((DT_H*(prices['as_'+p+'_da']@cda[p]+prices['as_'+p+'_rt']@(ca[p]-cda[p])) for p in products))
    cost_tou=DT_H*(tou@gi)
    m.setObjective(rev_ev+rev_e+rev_c-cost_tou-ncd_rate*ncd-pd_rate*pdpeak,GRB.MAXIMIZE)
    # Feasible no-battery/no-WM warm start; avoid repeated relaxation solves.
    counts=np.asarray(S.sum(axis=1)).ravel()
    x0=np.asarray(S.T@(targets/counts)).ravel()
    ev0=np.asarray(A@x0).ravel()/DT_H
    for vv in (ch,dc,chw,dcw,chn,dcn,da,actual,positive,negative,ecw,edw,nc,nd):
        vv.Start=np.zeros(n)
    for p in products:cda[p].Start=np.zeros(n);ca[p].Start=np.zeros(n)
    x.Start=x0;ev.Start=ev0;gi.Start=load-pv+ev0;soc.Start=np.full(n+1,.5)
    b.Start=np.zeros(n);be.Start=(ev0>=baseline).astype(float);bn.Start=np.zeros(n)
    ecn.Start=np.maximum(ev0-baseline,0);edn.Start=np.maximum(baseline-ev0,0)
    ncd.Start=max(0,float(np.max(load-pv+ev0)))
    pdpeak.Start=max(0,float(np.max((load-pv+ev0)[pd_mask]))) if pd_mask.any() else 0
    m.optimize()
    if not m.SolCount:
        raise RuntimeError(f'Oracle has no feasible solution: status={m.Status}')
    dep=sum(((-1 if p=='rd' else 1)*alpha[p]*ca[p].X for p in products))
    d=pd.DataFrame({'Interval start':index,'p_EV_kW':ev.X,'p_BESS_kW':ch.X-dc.X,
        'p_GI_kW':gi.X,'SOC':soc.X[1:],'baseline_kW':baseline,
        'p_load_kW':load,'p_PV_kW':pv,'p_DA_kW':da.X,
        'p_RT_deviation_kW':actual.X-da.X,'p_actual_kW':actual.X,
        'energy_obligation_kW':actual.X+dep,'P_EV_max_kW':max_ev,
        'p_up_bound_kW':up,'p_down_bound_kW':down,'TOU_$/kWh':tou,
        'BESS_charge_kW':ch.X,'BESS_discharge_kW':dc.X})
    for p in products:
        for label,val in [('DA',cda[p].X),('RT',ca[p].X-cda[p].X),('actual',ca[p].X)]:
            d[f'c_{p.upper()}_{label}_kW']=val
        d[f'alpha_{p.upper()}']=alpha[p]
    for k,a in prices.items():d['price_'+k]=a
    d['EV Revenue']=.4*DT_H*ev.X
    d['WM Energy Revenue']=DT_H*(prices['lmp_da']*da.X+prices['lmp_rt']*(actual.X-da.X+dep))
    d['WM Capacity Revenue']=sum((DT_H*(prices['as_'+p+'_da']*cda[p].X+prices['as_'+p+'_rt']*(ca[p].X-cda[p].X)) for p in products))
    d['WM Revenue']=d['WM Energy Revenue']+d['WM Capacity Revenue']
    d['TOU Cost']=-DT_H*tou*gi.X
    d['PD Cost']=0.;d['NCD Cost']=0.
    running_ncd=running_pd=0.
    for i in range(n):
        nxt=max(running_ncd,float(gi.X[i]),0)
        d.loc[i,'NCD Cost']=-ncd_rate*(nxt-running_ncd);running_ncd=nxt
        if pd_mask[i]:
            nxt=max(running_pd,float(gi.X[i]),0)
            d.loc[i,'PD Cost']=-pd_rate*(nxt-running_pd);running_pd=nxt
    components=['EV Revenue','WM Revenue','TOU Cost','PD Cost','NCD Cost']
    d['Total Revenue']=d[components].sum(axis=1)
    errors={'meter_max_kW':float(np.max(abs(gi.X-(load-pv+ev.X+ch.X-dc.X)))),
        'session_max_kWh':float(np.max(abs(S@x.X-targets))) if len(targets) else 0.,
        'soc_terminal_error':float(abs(soc.X[-1]-.5)),
        'accounting_error_dollar':float(abs(d['Total Revenue'].sum()-m.ObjVal))}
    summary={'method':'Full-horizon oracle; common pre-period baseline',
        'site':site,'start':start,'end_exclusive':end_exclusive,
        'status':int(m.Status),'accepted_1pct':bool(m.Status==GRB.OPTIMAL and m.MIPGap<=mip_gap+1e-8),
        'mip_gap':float(m.MIPGap),'runtime_s':float(m.Runtime),
        'objective_incumbent':float(m.ObjVal),'objective_upper_bound':float(m.ObjBound),
        'EV Energy kWh':float(DT_H*ev.X.sum()),**{c:float(d[c].sum()) for c in components+['WM Energy Revenue','WM Capacity Revenue','Total Revenue']},**errors}
    d.to_csv(output/'oracle_dispatch.csv',index=False)
    ev_meta.to_csv(output/'oracle_ev_sessions.csv',index=False)
    d.assign(Date=index.normalize()).groupby('Date')[components+['WM Energy Revenue','WM Capacity Revenue','Total Revenue']].sum().to_csv(output/'oracle_daily_summary.csv')
    pd.DataFrame([summary]).to_csv(output/'oracle_summary.csv',index=False)
    manifest={'baseline_definition':'fixed common pre-period history; no study-period feedback',
        'DA_information':'all awards optimized with full-week perfect information before first DA submission',
        'terminal_soc':.5,'initial_soc':.5,'initial_monthly_peaks':0,
        'scope':'single billing month; benchmark starts at billing-month start',
        'AS_clock':'naive timestamps preserved as California model clock; aware converted explicitly',
        'DA_envelope':'standard true-availability bounds',
        'source_MPC_sha256':hashlib.sha256((REPO/'upscaledev_imp_rollingMPC.ipynb').read_bytes()).hexdigest(),
        'ev_input':str(ev_file),'btm_input':str(btm_file),
        'note':'ObjBound is the solver upper bound. Incumbent dominance alone is not a proof. Comparator DA feasible envelopes must be checked.'}
    (output/'oracle_manifest.json').write_text(json.dumps(manifest,indent=2))
    if max(errors.values())>1e-3:
        raise RuntimeError(f'Oracle residual check failed: {errors}')
    return summary, d


def validate_weekly_oracle_saved(output_dir, runtime):
    """Recompute physical, tariff, and accounting checks from saved dispatch."""
    output = Path(output_dir)
    d = pd.read_csv(output/'oracle_dispatch.csv')
    idx = pd.DatetimeIndex(pd.to_datetime(d['Interval start']))
    P,E = float(runtime['P_BESS_max']),float(runtime['C_BESS'])
    lo,hi = float(runtime['SOC_BESS_min']),float(runtime['SOC_BESS_max'])
    eff = np.sqrt(float(runtime['gamma']))
    au = d[['c_RU_actual_kW','c_SP_actual_kW','c_NSP_actual_kW']].sum(axis=1).to_numpy()
    rd = d['c_RD_actual_kW'].to_numpy()
    actual = d['p_actual_kW'].to_numpy()
    up,down = d['p_up_bound_kW'].to_numpy(),d['p_down_bound_kW'].to_numpy()
    soc = d['SOC'].to_numpy()
    ch,dc = d['BESS_charge_kW'].to_numpy(),d['BESS_discharge_kW'].to_numpy()
    throughput = DT_H*(ch+dc).reshape(-1,96).sum(axis=1)
    dep = sum(((-1 if p=='RD' else 1)*d['alpha_'+p].to_numpy()*d['c_'+p+'_actual_kW'].to_numpy() for p in ('RU','RD','SP','NSP')))
    da_cols = ['p_DA_kW']+['c_'+p+'_DA_kW' for p in ('RU','RD','SP','NSP')]
    violations = {
        'actual_up_violation_kW': float(np.maximum(np.maximum(actual,0)+au-up,0).max()),
        'actual_down_violation_kW': float(np.maximum(np.maximum(-actual,0)+rd-down,0).max()),
        'actual_capacity_negative_violation_kW': max(0.,-min(d['c_'+p+'_actual_kW'].min() for p in ('RU','RD','SP','NSP'))),
        'SOC_lower_duration_violation': float(np.maximum(lo+2*DT_H/(E*eff)*au-soc,0).max()),
        'SOC_upper_duration_violation': float(np.maximum(soc-hi+2*DT_H*eff/E*rd,0).max()),
        'SOC_dynamics_error': float(np.max(abs(soc-np.r_[.5,soc[:-1]]-DT_H/E*(eff*ch-dc/eff)))),
        'daily_throughput_violation_kWh': max(0.,float(throughput.max()-2*E*(hi-lo))),
        'hourly_DA_award_variation_kW': max(float(np.ptp(d[c].to_numpy().reshape(-1,4),axis=1).max()) for c in da_cols),
        'obligation_identity_error_kW': float(np.max(abs(d['energy_obligation_kW']-actual-dep))),
        'BESS_simultaneous_charge_discharge_kW': float(np.minimum(ch,dc).max()),
        'BESS_power_violation_kW': max(0.,float(max(ch.max(),dc.max())-P)),
        'meter_identity_error_kW': float(np.max(abs(d['p_GI_kW']-d['p_load_kW']+d['p_PV_kW']-d['p_EV_kW']-ch+dc))),
        'terminal_SOC_error':float(abs(soc[-1]-.5)),
    }
    if runtime.get('Direction_Aligned_AddOn',True):
        q=d['energy_obligation_kW'].to_numpy()
        violations['direction_alignment_violation_kW']=max(float(au[q < -1e-6].max()) if (q < -1e-6).any() else 0.,float(rd[q>1e-6].max()) if (q>1e-6).any() else 0.)
    checks=pd.DataFrame([violations])
    checks['physical_validation_pass']=bool(max(violations.values())<1e-5)
    checks.to_csv(output/'oracle_validation.csv',index=False)
    summary=pd.read_csv(output/'oracle_summary.csv')
    for k,v in violations.items():summary[k]=v
    summary['physical_validation_pass']=bool(max(violations.values())<1e-5)
    summary.to_csv(output/'oracle_summary.csv',index=False)
    daily=pd.DataFrame({'Date':idx[::96].date,'BESS throughput kWh':throughput,'Daily throughput allowance kWh':2*E*(hi-lo)})
    daily.to_csv(output/'oracle_daily_throughput.csv',index=False)
    if not bool(checks['physical_validation_pass'].iloc[0]):
        raise ValueError(f'Oracle validation failed: {violations}')
    return checks


def rerun_corrected_standard_oracle(runtime, study_root):
    """Corrected model-clock prices and standard DA-envelope inclusion audit.
    Only oracle outputs are replaced; matched MPC results remain untouched.
    """
    root=Path(study_root)
    index=pd.date_range('2025-03-01','2025-03-08',freq='15min',inclusive='left')
    baseline=pd.read_csv(root/'inputs/fixed_oracle_baseline.csv')
    baseline=pd.Series(baseline['baseline_kW'].to_numpy(float),index=pd.to_datetime(baseline['Interval start'])).reindex(index)
    prices=_weekly_prices_exact_main(index)
    # Independent source-main parser extraction prevents timezone interpretation drift.
    import ast
    nb=json.loads((REPO/'upscaledev_imp_rollingMPC.ipynb').read_text())
    loop=''.join(nb['cells'][11]['source'])
    tree=ast.parse(loop)
    parser=next(x for x in tree.body if isinstance(x,ast.FunctionDef) and x.name=='parse_as_model_clock')
    ns={'pd':pd};exec(compile(ast.Module(body=[parser],type_ignores=[]),'source_AS_parser','exec'),ns)
    price_rows=[]
    for market,folder in [('da','AS_DAM'),('rt','AS_RTM')]:
        raw=pd.read_csv(REPO/'2025Data'/folder/'AS_price_2025_clear.csv')
        raw['datetime']=ns['parse_as_model_clock'](raw['datetime'])
        selected=raw[(raw['datetime']>=index[0])&(raw['datetime']<index[-1]+pd.Timedelta(minutes=15))]
        for p,col in [('ru','RegUp'),('rd','RegDown'),('sp','Spin'),('nsp','NonSpin')]:
            a=selected[col].to_numpy(float)
            expected=np.repeat(a,4)*.001 if market=='da' else a*.001/DT_H
            error=float(np.max(abs(expected-prices[f'as_{p}_{market}'])))
            if error>1e-12:raise ValueError('Oracle/source AS mismatch')
            price_rows.append({'input':f'as_{p}_{market}','max_error':error,'PASS':True})
    A,S,ub,meta=_weekly_ev_inputs(REPO/'2025Data/Site_Data_2025/NTPLL_EV_2025_QC.csv',index)
    P=float(runtime['P_BESS_max'])
    up=P+baseline.to_numpy();down=np.maximum(P-baseline.to_numpy()+np.asarray(A@ub).ravel()/DT_H,0)
    include=[]
    for case in ('benchmark_persistence','benchmark_perfect'):
        paths=sorted((root/'cases'/case/'Validation_Traces').rglob('rolling_validation_trace_*.csv'))
        trace=pd.concat([pd.read_csv(p) for p in paths],ignore_index=True).sort_values('Interval start')
        if not np.array_equal(pd.to_datetime(trace['Interval start']).to_numpy(),index.to_numpy()):raise ValueError('Comparator time mismatch')
        for key,col in [('lmp_da','LMP_DA_$/kWh'),('lmp_rt','LMP_RT_$/kWh')]:
            error=float(np.max(abs(prices[key]-trace[col].to_numpy(float))))
            price_rows.append({'input':case+'_'+key,'max_error':error,'PASS':error<1e-8})
            if error>1e-8:raise ValueError('Oracle/main LMP mismatch')
        tou=np.asarray(runtime['get_sdge_dgr_tariff_profile'](index)[0])
        terror=float(np.max(abs(tou-trace['TOU_$/kWh'].to_numpy(float))))
        if terror>1e-8:raise ValueError('Oracle/main TOU mismatch')
        price_rows.append({'input':case+'_TOU','max_error':terror,'PASS':True})
        da=trace['p_DA_kW'].to_numpy(float)
        cu=trace[['c_RU_DA_kW','c_SP_DA_kW','c_NSP_DA_kW']].sum(axis=1).to_numpy(float)
        crd=trace['c_RD_DA_kW'].to_numpy(float)
        need_up=np.maximum(da,cu);need_down=np.maximum(-da,crd)
        actual=trace['p_actual_kW'].to_numpy(float)
        ca=trace[['c_RU_actual_kW','c_SP_actual_kW','c_NSP_actual_kW']].sum(axis=1).to_numpy(float)
        cad=trace['c_RD_actual_kW'].to_numpy(float)
        include.append({'case_id':case,
            'baseline_max_error_kW':float(np.max(abs(trace['baseline_kW'].to_numpy()-baseline.to_numpy()))),
            'EV_energy_error_kWh':float(abs(.25*trace['p_EV_kW'].sum()-meta['target_kwh'].sum())),
            'standard_DA_up_violation_kW':max(0.,float((need_up-up).max())),
            'standard_DA_down_violation_kW':max(0.,float((need_down-down).max())),
            'RT_up_violation_kW':max(0.,float((np.maximum(actual,0)+ca-up).max())),
            'RT_down_violation_kW':max(0.,float((np.maximum(-actual,0)+cad-down).max())),
            'terminal_SOC_error':float(abs(trace['SOC'].iloc[-1]-.5))})
    pd.DataFrame(price_rows).to_csv(root/'oracle_price_definition_validation.csv',index=False)
    pd.DataFrame(include).to_csv(root/'oracle_comparator_inclusion.csv',index=False)
    standard,_=run_weekly_oracle(runtime,baseline,root/'oracle',overwrite=True,time_limit_s=1800,threads=2)
    validate_weekly_oracle_saved(root/'oracle',runtime)
    return pd.DataFrame([standard]),pd.DataFrame(include)

def export_oracle_comparison_tables(study_root):
    """Export the standard full-horizon benchmark only."""
    root=Path(study_root);tables=root/'tables';tables.mkdir(parents=True,exist_ok=True)
    rows=[];days=[]
    for case_id,label in [('oracle','Full-horizon oracle')]:
        row=pd.read_csv(root/case_id/'oracle_summary.csv').iloc[0].to_dict()
        row['case_id']=case_id;row['forecast']=label
        row['claim_scope']='Full-information benchmark with true-availability DA offer bounds; not a universal feasible-set upper bound for forecast-driven DA commitments'
        row['absolute_solver_gap_USD']=float(row['objective_upper_bound']-row['objective_incumbent'])
        rows.append(row)
        d=pd.read_csv(root/case_id/'oracle_daily_summary.csv')
        d['case_id']=case_id;d['forecast']=label;days.append(d)
    weekly=pd.DataFrame(rows);daily=pd.concat(days,ignore_index=True)
    weekly.to_csv(tables/'oracle_financial_comparison.csv',index=False)
    daily.to_csv(tables/'oracle_daily_comparison.csv',index=False)
    claims={
        'prices':'All AS, LMP and TOU arrays audited against current source MPC parsing and executed prices.',
        'baseline':'Common exogenous baseline calculated exclusively from pre-March history.',
        'initialization':'All begin before first DA submission with 50% SOC and zero March billing peaks.',
        'standard_oracle':'Not a universal upper bound for forecast-driven DA awards; observed Persistence DA-down offer exceeds true-availability envelope by73.216kW.',
        'incumbent_vs_bound':'Plotted net revenue is a feasible oracle incumbent. Certified upper bound is separately provided; exact optimal value lies between them.',
        'tolerance':'The oracle solve satisfies the user-requested1% MIPGap; small economic differences are not proofs of exact-optimum separation.'
    }
    (tables/'oracle_comparison_claims.json').write_text(json.dumps(claims,indent=2))
    return weekly,daily

def run_weekly_oracle_benchmark(study_root=None, overwrite=False):
    """User-facing reproducible entry point; existing results are protected by default.
    Rebuilds the common reference from saved pre-period history, validates both
    completed matched MPC cases, and preserves the standard true-availability DA envelopes.
    No historical oracle output is silently reused as a fresh optimization.
    """
    import os
    root=Path(study_root or REPO/'Sensitivity_Results/Rolling_24h/site_week_20250301_07').resolve()
    if not overwrite:
        existing=[root/c/'oracle_summary.csv' for c in ('oracle',) if (root/c/'oracle_summary.csv').exists()]
        if existing:
            raise FileExistsError('Oracle results already exist. Review them or explicitly set ORACLE_OVERWRITE=True before rerunning: '+str(existing))
    manifests={}
    for case,forecast in [('benchmark_persistence','Persistence'),('benchmark_perfect','Perfect')]:
        p=root/'cases'/case/'COMPLETE.json'
        if not p.exists():raise FileNotFoundError(f'Complete matched MPC case required: {p}')
        m=json.loads(p.read_text());manifests[case]=m
        expected_dates=[f'2025-03-{d:02d}' for d in range(1,8)]
        if m.get('forecast')!=forecast or m.get('dates')!=expected_dates or m.get('baseline_policy')!='fixed_preperiod':
            raise ValueError(f'Wrong matched-benchmark configuration: {case}')
        if not m.get('validation',{}).get('PASS',False):
            raise ValueError(f'Matched MPC case did not pass validation: {case}')
        for key,expected in [('initial_SOC',.5),('terminal_SOC',.5),('eta',1.),('P_BESS_kW',250.),('E_BESS_kWh',332.),('alpha_SP',.2),('alpha_NSP',.2)]:
            if not np.isclose(float(m[key]),expected):raise ValueError(f'Unexpected {key}: {case}')
    history=root/'inputs/common_perfect_preperiod_dispatch.csv'
    saved=pd.read_csv(root/'inputs/fixed_oracle_baseline.csv')
    index=pd.DatetimeIndex(pd.to_datetime(saved['Interval start']))
    expected_index=pd.date_range('2025-03-01','2025-03-09',freq='15min',inclusive='left')
    if not np.array_equal(index.to_numpy(),expected_index.to_numpy()):
        raise ValueError('Shared fixed baseline must cover study week and one look-ahead day')
    rebuilt=fixed_preperiod_baseline(history,index)
    if np.max(abs(rebuilt.to_numpy()-saved['baseline_kW'].to_numpy(float)))>1e-8:
        raise ValueError('Saved fixed baseline disagrees with its pre-period history')
    runtime={'__name__':'weekly_oracle_runtime'}
    source=json.loads((REPO/'upscaledev_imp_rollingMPC.ipynb').read_text())
    previous_cwd=Path.cwd()
    try:
        os.chdir(REPO)
        for cell in (3,5):
            exec(compile(''.join(source['cells'][cell]['source']),f'source_MPC_cell_{cell}','exec'),runtime)
        runtime['AS_ACTIVATION_MODE']='caiso_regulation'
        runtime['AS_ACTIVATION_CONSTANTS'].update({'SP':.2,'NSP':.2})
        runtime['AS_CONTINGENCY_ACTIVATION_FILE']=None
        runtime['AS_ACTIVATION_SCALE']={k:1. for k in ('RU','RD','SP','NSP')}
        if not np.isclose(runtime['P_BESS_max'],250.) or not np.isclose(runtime['C_BESS'],332.):
            raise ValueError('Current source BESS settings do not match the completed benchmark')
        rerun_corrected_standard_oracle(runtime,root)
        weekly,daily=export_oracle_comparison_tables(root)
    finally:
        os.chdir(previous_cwd)
    return weekly,daily


## Run the standard weekly oracle benchmark
Set RUN_WEEKLY_ORACLE=True below only when a new optimization is intended. Existing results are protected unless ORACLE_OVERWRITE=True. The benchmark retains actual-availability DA limits and is not a universal upper bound for forecast-based DA awards.


In [ ]:
RUN_WEEKLY_ORACLE = False
ORACLE_OVERWRITE = False
ORACLE_STUDY_ROOT = REPO / 'Sensitivity_Results/Rolling_24h/site_week_20250301_07'

if RUN_WEEKLY_ORACLE:
    oracle_weekly_table, oracle_daily_table = run_weekly_oracle_benchmark(
        study_root=ORACLE_STUDY_ROOT, overwrite=ORACLE_OVERWRITE)
    display(oracle_weekly_table[['forecast', 'Total Revenue', 'WM Revenue',
                                'TOU Cost', 'PD Cost', 'NCD Cost', 'EV Revenue',
                                'objective_upper_bound', 'mip_gap']])
else:
    print('Oracle optimization is OFF. Set RUN_WEEKLY_ORACLE=True to run; existing results are protected unless ORACLE_OVERWRITE=True.')
